# Heisenberg Evolution and Operator Entanglement in a Spin Chain using MPO

(Matrix Product Operators) with bond dimension χ controlled by operator entanglement.



In [10]:
using ITensors, ITensorMPS

Evolution of an operator in the Heisenberg representation :
$$O(t)=U^\dagger(t) O U(t)$$

with $U(t) = e^{-iHt}$

---

let's creat a spin chain of $6$, $1/2$ particules.

In [11]:
# nombre de spins
N = 6

# creation des sites
sites = ITensors.siteinds("S=1/2", N)

println(sites)

Index{Int64}[(dim=2|id=62|"S=1/2,Site,n=1"), (dim=2|id=433|"S=1/2,Site,n=2"), (dim=2|id=566|"S=1/2,Site,n=3"), (dim=2|id=864|"S=1/2,Site,n=4"), (dim=2|id=190|"S=1/2,Site,n=5"), (dim=2|id=23|"S=1/2,Site,n=6")]


---

Here we will take the Hamiltonian :

$$H = \sum_j ( X_j X_{j+1} + gX_j + hZ_j )$$

In [12]:
g = 0.5
h = 0.5

function compute_H(g::Float64=0.5, h::Float64=0.5)::MPO
  ampo = AutoMPO()

  for j=1:N-1
      add!(ampo,"X",j,"X",j+1) # interaction XX
  end

  for j=1:N
      add!(ampo, g,"X",j) # champ X
      add!(ampo, h,"Z",j) # champ Z
  end

  return MPO(ampo, sites)
end

H = compute_H(g,h)

println("max bond dimension : ", maxlinkdim(H))

max bond dimension : 3


bond dimension faible, $3$.

La matrice complète serait : $2^N x 2^N$

donc pour $N=6$ : $64 x 64$

Mais le MPO reste très compact car l'information doit seulement se propager localement dans le MPO pour cet hamiltinien.

---

On cherche maintenant a faire evoluer notre opérateur $\hat{O}$, calculons donc l'opérateur évolution $U = e^{-iHt}$.

calculer $U$ pour $N$ trop grand est trop couteux, on va chercher a simplifier en utilisant : **Trotter decomposition**.

$$H = \sum_j h_{j,j+1} \quad \text{avec ici} \quad h_{j,j+1} = X_jX_{j+1} +\frac{g}{2}( X_j+X_{j+1})+\frac{h}{2}( Z_j+Z_{j+1})$$

On a donc pour un petit pas de temps :
$$U(\tau) = e^{-iH\tau} \approx \prod_j e^{-i\tau h_{j,j+1}} \approx \prod_{j _{odd}}e^{-i\tau h_{j,j+1}/2} \prod_{j _{even}}e^{-i\tau h_{j,j+1}} \prod_{j _{odd}}e^{-i\tau h_{j,j+1}/2}$$

Chaque terme : $e^{-i\tau h_{j,j+1}}$ agit seulement sur 2 spins.

Donc on peut le représenter comme un tensor à 4 indices.

C’est ce qu’on appelle une **gate TEBD**.

In [20]:
cutoff = 1e-8
τ = 0.1

# Hamiltonien local h_{j,j+1}
function build_two_site_hamiltonian(j::Int64)
  s1 = sites[j]
  s2 = sites[j + 1]
  hj =
     op("X", s1) * op("X", s2) +
     (g/2) * (op("X", s1) * op("Id", s2) + op("Id", s1) * op("X", s2)) +
     (h/2) * (op("Z", s1) * op("Id", s2) + op("Id", s1) * op("Z", s2))
  return hj
end

# Construction des gates TEBD
function build_gates(τ::Float64)::Tuple{Vector{ITensor}, Vector{ITensor}}
  gates_odd = ITensor[]
  gates_even = ITensor[]
  for j in 1:(N - 1)
      hj = build_two_site_hamiltonian(j)
      Gj = exp(-im * τ / 2 * hj) # gate Trotter ordre 2
      if isodd(j)
        push!(gates_odd, Gj)
      else
        push!(gates_even, Gj)
      end
  end
  return gates_odd, gates_even
end

# Application U† O U pour une liste de gates
function apply_heisenberg_step(O::MPO, gates::Vector{ITensor})::MPO
  O = apply(gates, O; cutoff=cutoff)
   O = apply(dag.(reverse(gates)), O; cutoff=cutoff) # apply_dag=true car on souhaite appliqué en bas et va automatiquement prendre le dag(gates)
  return O
end

# Evolution temporelle
function simulate_operator_evolution(O::MPO, steps::Int64, τ::Float64=0.01, show_maxbond::Bool=false)::MPO
  gates_odd, gates_even = build_gates(τ)

  for step in 1:steps
    O = apply_heisenberg_step(O,gates_odd)
    O = apply_heisenberg_step(O,gates_even)
    O = apply_heisenberg_step(O,gates_odd)
    if show_maxbond
      println("step ", step, ": χ=", maxlinkdim(O))
    end
  end
  return O
end

simulate_operator_evolution (generic function with 3 methods)

---

## Check stability of the simulation

If $O = I$ we have :

$$U^\dagger I U = I \qquad \forall U$$

let's check

In [14]:
# construction MPO identite
IdMPO = MPO(sites, "Id")

6-element MPO:
 ((dim=2|id=62|"S=1/2,Site,n=1")', (dim=2|id=62|"S=1/2,Site,n=1"), (dim=1|id=81|"Link,l=1"))
 ((dim=2|id=433|"S=1/2,Site,n=2")', (dim=2|id=433|"S=1/2,Site,n=2"), (dim=1|id=431|"Link,l=2"), (dim=1|id=81|"Link,l=1"))
 ((dim=2|id=566|"S=1/2,Site,n=3")', (dim=2|id=566|"S=1/2,Site,n=3"), (dim=1|id=559|"Link,l=3"), (dim=1|id=431|"Link,l=2"))
 ((dim=2|id=864|"S=1/2,Site,n=4")', (dim=2|id=864|"S=1/2,Site,n=4"), (dim=1|id=591|"Link,l=4"), (dim=1|id=559|"Link,l=3"))
 ((dim=2|id=190|"S=1/2,Site,n=5")', (dim=2|id=190|"S=1/2,Site,n=5"), (dim=1|id=284|"Link,l=5"), (dim=1|id=591|"Link,l=4"))
 ((dim=2|id=23|"S=1/2,Site,n=6")', (dim=2|id=23|"S=1/2,Site,n=6"), (dim=1|id=284|"Link,l=5"))

In [15]:
println("Max bond dimension = ", maxlinkdim(IdMPO))

Max bond dimension = 1


max bond dimension $= 1 \implies \chi = 1$, it's the simpliest MPO possible.

In [21]:
# Simulation
steps = 20
Ot = simulate_operator_evolution(IdMPO, steps, τ)

println("Max bond dimension = ", maxlinkdim(Ot))
println("\nIdentity distance = ", norm(Ot - IdMPO))

Max bond dimension = 1

Identity distance = 8.697041074470232e-9


On retrouve bien le resultat attendu :
$$U^{\dagger}IU = I$$
Apres l'application de l'évolution on retombe sur l'identité, a une erreur numerique pret de l'ordre de $10^{-9}$.

---

## Local operator $Z_3$

$$Z_3 = \mathbb{I} \otimes \mathbb{I} \otimes Z \otimes \mathbb{I} \otimes \mathbb{I} \otimes \mathbb{I}$$

In [22]:
j = 3
# Create an MPO with identity operators on all sites
O = MPO(sites, "Id")
# Get the Z operator for site j
Zj_op = op("Z", sites[j])
# Replace the Identity operator at site j with the Z operator
O[j] = Zj_op

Ot = simulate_operator_evolution(O, steps, τ, true)

println("Max bond dimension = ", maxlinkdim(Ot))

step 1: χ=1
step 2: χ=1
step 3: χ=1
step 4: χ=1
step 5: χ=1
step 6: χ=1
step 7: χ=1
step 8: χ=1
step 9: χ=1
step 10: χ=1
step 11: χ=1
step 12: χ=1
step 13: χ=1
step 14: χ=1
step 15: χ=1
step 16: χ=1
step 17: χ=1
step 18: χ=1
step 19: χ=1
step 20: χ=1
Max bond dimension = 1


---

References

- [Julia documentation](https://docs.julialang.org/en/v1/)
- [ITensor documentation](https://docs.itensor.org/ITensors/stable/index.html)